In [2]:
import sys
print(sys.executable)

c:\Users\Ravichandran\OneDrive\Desktop\Ml Alogiritms dally progress\.venv\Scripts\python.exe


In [3]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\Ravichandran\OneDrive\Desktop\fir crime type prediction model\model_training
['01_train_basline_tfidf_logreg.ipynb']


In [4]:
import os
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
import ast
crime = pd.read_csv(r"C:\Users\Ravichandran\OneDrive\Desktop\dataset_A_clean.csv")

crime['text'] = crime['facts'].astype(str)
crime['labels'] = crime['final_labels'].apply(ast.literal_eval)

In [6]:
type(crime['labels'][0])

list

In [7]:
crime.shape

(1200, 18)

In [8]:
crime.columns.to_list()

['ipc_sections',
 'crime_type',
 'facts',
 'legal_issues',
 'judgment_reason',
 'summary',
 'region',
 'facts_lang',
 'facts_token_len',
 'ipc_list',
 'ipc_count',
 'ipc_based_labels',
 'final_labels',
 'summary_len',
 'reason_len',
 'legal_issues_len',
 'text',
 'labels']

In [9]:
# ignoring empty labels rows only for training 

crime_train = crime[crime['labels'].apply(len)>0]

In [11]:
crime_train.head()

,ipc_sections,crime_type,facts,legal_issues,judgment_reason,summary,region,facts_lang,facts_token_len,ipc_list,ipc_count,ipc_based_labels,final_labels,summary_len,reason_len,legal_issues_len,text,labels
0,"['120B', '121', '121A']",Narcotics,Jibangshu Paul was apprehended carrying Rs. 32...,"['Whether fresh bail is needed when new, more ...",The court held that newly added serious UA(P) ...,Bail earlier granted to Jibangshu Paul under I...,Assam,en,55,"['120B', '121', '121A']",3,['homicide'],['homicide'],28,36,239,Jibangshu Paul was apprehended carrying Rs. 32...,[homicide]
1,"['376', '354', '343', '109', '220', '348', '33...",Sexual Offense,The case involves custodial rape of a woman by...,['Whether suspension of sentence requires deta...,The court held that due to the grave nature of...,Madras High Court cancelled the bail granted t...,Tamil Nadu,en,52,"['376', '354', '343', '109', '220', '348', '33...",8,"['sexual_offence', 'assault']","['sexual_offence', 'assault']",29,53,244,The case involves custodial rape of a woman by...,"[sexual_offence, assault]"
2,"['465', '468', '471', '474', '420', '511', '34']",Fraud or Cheating,"Hyderali, a government contractor, was accused...",['Whether anticipatory bail can be granted whe...,The court ruled that despite other co-accused ...,Kerala High Court rejected anticipatory bail f...,Kerala,en,54,"['465', '468', '471', '474', '420', '511', '34']",7,"['fraud', 'cheating']","['fraud', 'cheating']",29,56,285,"Hyderali, a government contractor, was accused...","[fraud, cheating]"
3,"['326', '307', '120B', '201']",Others,"The petitioner, a government employee, alleged...",['Whether bail should be cancelled under Secti...,The court found no cogent or overwhelming reas...,Calcutta High Court refused to cancel bail of ...,West Bengal,en,49,"['326', '307', '120B', '201']",4,"['grievous_hurt', 'weapon_used', 'homicide']","['grievous_hurt', 'weapon_used', 'homicide']",27,40,304,"The petitioner, a government employee, alleged...","[grievous_hurt, weapon_used, homicide]"
4,"['302', '34']",Murder,Shankri Devi and co-accused were charged with ...,['Whether the court’s earlier bail rejection o...,The court held that its earlier rejection of b...,Jammu & Kashmir High Court rejected the review...,Jammu & Kashmir,en,49,"['302', '34']",2,['homicide'],['homicide'],29,42,256,Shankri Devi and co-accused were charged with ...,[homicide]


In [13]:
crime_train.shape

(1040, 18)

In [16]:
crime_train['labels'].value_counts()

labels
[fraud, cheating]                                                                                      96
[fraud, cheating, cyber_crime]                                                                         91
[domestic_violence, homicide]                                                                          81
[theft, robbery]                                                                                       44
[kidnapping, sexual_offence]                                                                           41
                                                                                                       ..
[house_trespass, domestic_violence, weapon_used, assault, theft]                                        1
[house_trespass, domestic_violence, assault]                                                            1
[fraud, house_trespass, cheating]                                                                       1
[sexual_offence, fraud, kidnapping, wea

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

primary_label = crime_train["labels"].apply(lambda x: x[0] if len(x) > 0 else "NONE")

X_train, X_test, y_train, y_test = train_test_split(
    crime_train["text"],
    crime_train["labels"],
    test_size=0.2,
    random_state=42,
    stratify= primary_label
)

mlb = MultiLabelBinarizer()
y_train_bin = mlb.fit_transform(y_train)
y_test_bin = mlb.transform(y_test)


In [32]:
print('Features:',crime_train['text'][:10], '\nLabels:', crime_train['labels'][:10], sep='\n')
print("crime_train shape:", crime_train.shape)
print("crime_train shape:", crime_train['labels'].shape)

Features:
0     Jibangshu Paul was apprehended carrying Rs. 32...
1     The case involves custodial rape of a woman by...
2     Hyderali, a government contractor, was accused...
3     The petitioner, a government employee, alleged...
4     Shankri Devi and co-accused were charged with ...
5     Hardeep Singh @ Dipi, a juvenile, was implicat...
6     Bharath Kumar was accused of conspiring to mur...
7     Petitioners, police personnel, were accused of...
8     Aslam Babalal Desai was arrested in connection...
10    Ishtiaq Hasan Khan was accused in a public day...
Name: text, dtype: object

Labels:
0                                 [homicide]
1                  [sexual_offence, assault]
2                          [fraud, cheating]
3     [grievous_hurt, weapon_used, homicide]
4                                 [homicide]
5           [homicide, weapon_used, robbery]
6                        [assault, homicide]
7                        [assault, homicide]
8                        [assault, 

In [ ]:
import joblib
joblib.dump(mlb, "artifacts/mlb.pkl")

['artifacts/mlb.pkl']

In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
    max_features=50000,
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

joblib.dump(vectorizer, "artifacts/tfidf.pkl")


['artifacts/tfidf.pkl']

In [44]:
print(X_train_vec)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 58051 stored elements and shape (832, 4515)>
  Coords	Values
  (0, 2723)	0.13955590259002065
  (0, 392)	0.07561678772128988
  (0, 4431)	0.03872426967329228
  (0, 3605)	0.09751474645666068
  (0, 1074)	0.04756661556469011
  (0, 217)	0.02781909954719029
  (0, 357)	0.045784909629269344
  (0, 1179)	0.12170217912170056
  (0, 4081)	0.06677310682278112
  (0, 1585)	0.10304777287052576
  (0, 3487)	0.07360081233982975
  (0, 1272)	0.09519509246742935
  (0, 1779)	0.04756661556469011
  (0, 3576)	0.12723520553556564
  (0, 2805)	0.02710828707953623
  (0, 2219)	0.11360622165764321
  (0, 904)	0.08440589304881871
  (0, 2095)	0.13955590259002065
  (0, 1534)	0.1306290408558606
  (0, 1446)	0.13464309606334904
  (0, 2911)	0.11536846992498077
  (0, 1626)	0.09834565072676339
  (0, 1524)	0.12429533165914083
  (0, 3810)	0.12723520553556564
  (0, 4362)	0.1859858057968828
  :	:
  (831, 2885)	0.13494802603858838
  (831, 4253)	0.11289394985658238
  (831, 

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

model = OneVsRestClassifier(LogisticRegression(max_iter=1000, n_jobs=-1))
model.fit(X_train_vec, y_train_bin)

joblib.dump(model, "artifacts/crime_model_v1.pkl")


['artifacts/crime_model_v1.pkl']

In [46]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_vec)
print(classification_report(y_test_bin, y_pred, target_names=mlb.classes_))


                       precision    recall  f1-score   support

              assault       0.00      0.00      0.00        40
             cheating       1.00      0.44      0.61        48
criminal_intimidation       0.00      0.00      0.00        31
          cyber_crime       1.00      0.29      0.44        21
    domestic_violence       0.96      0.59      0.73        44
            extortion       0.00      0.00      0.00        33
                fraud       0.95      0.57      0.71        74
        grievous_hurt       0.00      0.00      0.00         8
             homicide       0.96      0.37      0.53        73
       house_trespass       0.00      0.00      0.00        13
           kidnapping       1.00      0.25      0.40        36
              robbery       1.00      0.06      0.11        36
       sexual_offence       1.00      0.17      0.29        30
                theft       1.00      0.06      0.12        32
          weapon_used       0.00      0.00      0.00  

c:\Users\Ravichandran\OneDrive\Desktop\Ml Alogiritms dally progress\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ravichandran\OneDrive\Desktop\Ml Alogiritms dally progress\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [52]:
y_prob = model.predict_proba(X_test_vec)  # list of arrays, one per label
import numpy as np
y_prob = np.vstack(y_prob) # shape: (n_samples, n_labels)


In [53]:
threshold = 0.25
y_pred_th = (y_prob >= threshold).astype(int)


In [54]:
from sklearn.metrics import classification_report

print(classification_report(y_test_bin, y_pred_th, target_names=mlb.classes_, zero_division=0))

                       precision    recall  f1-score   support

              assault       0.53      0.62      0.57        40
             cheating       0.73      0.96      0.83        48
criminal_intimidation       0.31      0.55      0.40        31
          cyber_crime       1.00      0.67      0.80        21
    domestic_violence       0.91      0.98      0.95        44
            extortion       0.84      0.79      0.81        33
                fraud       0.51      0.91      0.65        74
        grievous_hurt       0.00      0.00      0.00         8
             homicide       0.67      0.93      0.78        73
       house_trespass       0.00      0.00      0.00        13
           kidnapping       0.97      0.78      0.86        36
              robbery       0.88      0.61      0.72        36
       sexual_offence       0.85      0.77      0.81        30
                theft       0.90      0.56      0.69        32
          weapon_used       0.00      0.00      0.00  

In [61]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
import numpy as np
from sklearn.metrics import classification_report

model_bal = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        n_jobs=-1,
        class_weight="balanced"
    )
)
model_bal.fit(X_train_vec, y_train_bin)

proba_list_bal = model_bal.predict_proba(X_test_vec)  # len = 208, each (15,)
y_prob_bal = np.array(proba_list_bal)                 # -> (208, 15)
y_pred_bal = (y_prob_bal >= 0.25).astype(int)

print(y_test_bin.shape)   # (208, 15)
print(y_pred_bal.shape)   # (208, 15)

print(classification_report(
    y_test_bin,
    y_pred_bal,
    target_names=mlb.classes_,
    zero_division=0
))

print("y_test_bin:", y_test_bin.shape)
print("raw proba_list_bal len:", len(proba_list_bal))
print("proba_list_bal[0] shape:", proba_list_bal[0].shape)
print("y_prob_bal:", y_prob_bal.shape)
print("y_pred_bal:", y_pred_bal.shape)




(208, 15)
(208, 15)
                       precision    recall  f1-score   support

              assault       0.21      1.00      0.35        40
             cheating       0.32      1.00      0.48        48
criminal_intimidation       0.17      0.90      0.28        31
          cyber_crime       0.48      1.00      0.65        21
    domestic_violence       0.66      1.00      0.79        44
            extortion       0.25      1.00      0.40        33
                fraud       0.41      0.99      0.58        74
        grievous_hurt       0.22      1.00      0.36         8
             homicide       0.39      0.99      0.56        73
       house_trespass       0.11      0.69      0.19        13
           kidnapping       0.31      1.00      0.48        36
              robbery       0.30      1.00      0.46        36
       sexual_offence       0.26      1.00      0.41        30
                theft       0.24      1.00      0.39        32
          weapon_used       0.13  

In [ ]:
label_names = list(mlb.classes_)
print(label_names)
print(len(label_names))


['assault', 'cheating', 'criminal_intimidation', 'cyber_crime', 'domestic_violence', 'extortion', 'fraud', 'grievous_hurt', 'homicide', 'house_trespass', 'kidnapping', 'robbery', 'sexual_offence', 'theft', 'weapon_used']
15


In [76]:
label_names = list(mlb.classes_)  # ['assault', 'cheating', ..., 'weapon_used']

LABEL_THRESHOLDS = {
    "homicide": 0.25,
    "sexual_offence": 0.25,
    "kidnapping": 0.30,
    "domestic_violence": 0.30,
    "cyber_crime": 0.35,
    "fraud": 0.40,
    "cheating": 0.40,
    "robbery": 0.40,
    "extortion": 0.40,
    "assault": 0.45,

    "theft": 0.55,
    "criminal_intimidation": 0.55,
    "house_trespass": 0.60,
    "weapon_used": 0.60,
    "grievous_hurt": 0.60,
}


In [ ]:

import numpy as np
y_pred_custom = np.zeros_like(y_prob_bal, dtype=int)

for j, label in enumerate(label_names):
    th = LABEL_THRESHOLDS[label]
    y_pred_custom[:, j] = (y_prob_bal[:, j] >= th).astype(int)

In [78]:
from sklearn.metrics import classification_report

print(classification_report( y_test_bin, y_pred_custom, target_names=label_names, zero_division=0 ))


                       precision    recall  f1-score   support

              assault       0.48      0.75      0.59        40
             cheating       0.66      0.96      0.78        48
criminal_intimidation       0.45      0.32      0.38        31
          cyber_crime       0.84      1.00      0.91        21
    domestic_violence       0.76      1.00      0.86        44
            extortion       0.61      0.94      0.74        33
                fraud       0.58      0.88      0.70        74
        grievous_hurt       1.00      0.25      0.40         8
             homicide       0.39      0.99      0.56        73
       house_trespass       0.00      0.00      0.00        13
           kidnapping       0.49      0.94      0.65        36
              robbery       0.54      0.89      0.67        36
       sexual_offence       0.26      1.00      0.41        30
                theft       0.83      0.59      0.69        32
          weapon_used       0.75      0.17      0.27  

In [79]:
import json

with open("artifacts/label_thresholds.json", "w") as f:
    json.dump(LABEL_THRESHOLDS, f, indent=2)


In [80]:
import os, json, joblib

os.makedirs("models/crime_type_v1", exist_ok=True)

joblib.dump(model_bal, "models/crime_type_v1/model.joblib")
joblib.dump(vectorizer, "models/crime_type_v1/vectorizer.joblib")
joblib.dump(mlb, "models/crime_type_v1/label_binarizer.joblib")

with open("models/crime_type_v1/label_thresholds.json", "w") as f:
    json.dump(LABEL_THRESHOLDS, f, indent=2)

metadata = {
    "trained_on": "Dataset_A_FIR_1200",
    "labels": len(mlb.classes_),
    "threshold_type": "per_label",
    "micro_f1": 0.63,
    "date": "2026-01-18",
}
with open("models/crime_type_v1/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)


In [4]:
import os, sys
print("CWD:", os.getcwd())
print("sys.path[0]:", sys.path[0])


CWD: c:\Users\Ravichandran\OneDrive\Desktop\fir crime type prediction model\model_training\models
sys.path[0]: C:\Users\Ravichandran\AppData\Local\Programs\Python\Python313\python313.zip


In [14]:
import os, sys, pkgutil

print("CWD:", os.getcwd())
print("First 5 sys.path entries:")
for p in sys.path[:5]:
    print("  ", p)

# Compute project_root the way we expect
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))
print("project_root guess:", project_root)

print("Contents of project_root:")
print(os.listdir(project_root))


CWD: c:\Users\Ravichandran\OneDrive\Desktop\fir crime type prediction model\model_training\models
First 5 sys.path entries:
   c:\Users\Ravichandran\OneDrive\Desktop\fir crime type prediction model
   C:\Users\Ravichandran\AppData\Local\Programs\Python\Python313\python313.zip
   C:\Users\Ravichandran\AppData\Local\Programs\Python\Python313\DLLs
   C:\Users\Ravichandran\AppData\Local\Programs\Python\Python313\Lib
   C:\Users\Ravichandran\AppData\Local\Programs\Python\Python313
project_root guess: c:\Users\Ravichandran\OneDrive\Desktop\fir crime type prediction model
Contents of project_root:
['.vscode', 'data', 'data_cleane_files', 'Data_pipeline', 'inference', 'model_training', 'Readme.md']


In [1]:
from inference.crime_predictor import CrimePredictor

cp = CrimePredictor(model_dir="../models/crime_type_v1")

t1 = ("My husband beats me every day and demands dowry from my parents and "
      "threatens to throw me out of the house if I refuse to bring more money.")
print(cp.predict(t1, mode="citizen"))



ModuleNotFoundError: No module named 'inference'

In [22]:
cp = CrimePredictor(model_dir="../models/crime_type_v1")

text = ("He entered the house at night without permission, threatened with a knife, "
        "and stole jewellery and cash from the locked cupboard.")
print(cp.predict(text, mode="citizen"))
print(cp.predict(text, mode="lawyer"))


{'labels': ['fraud', 'homicide', 'robbery'], 'confidence': 0.5761732792979859, 'warning': 'This is only an AI indication, not legal advice.'}
{'labels': ('extortion', 'fraud', 'homicide', 'robbery', 'theft'), 'confidence': 0.5761732792979859, 'warning': 'This is only an AI indication, not legal advice.'}
